# MLOps Project: Diabetes with Linear Regression

1. Install the libraries.
2. Load, inspect, and clean the data.
3. Split the data for training and evaluation.
4. Train a linear regression model and measure its results.
5. Track the experiment and model with MLflow.
6. Assign the `champion` alias.
7. Create an API with FastAPI and test its endpoints.
8. Create automated tests and files for GitHub Actions.

## 1. Install the libraries

In [1]:
%pip install -q "mlflow>=2.12,<4" fastapi uvicorn httpx pytest scikit-learn pandas numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.7/148.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1

## 2. Import the tools and prepare the folders

- `pandas` and `numpy`: work with data.
- `scikit-learn`: train the model.
- `mlflow`: track experiments and models.
- `fastapi`: create an API.
- `pytest`: run automated tests.

In [3]:
import os
import json
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

os.makedirs("data/raw", exist_ok=True)
os.makedirs("src", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs(".github/workflows", exist_ok=True)

# Local MLflow database
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Diabetes-Regression")

print("Environment ready.")

Environment ready.


## 3. Load the Diabetes data

`X` contains the features used for prediction. `y` is the target we want to estimate.

In [4]:
diabetes = load_diabetes(as_frame=True)
df_original = diabetes.frame.copy()

df_original.to_csv("data/raw/diabetes.csv", index=False)

print("Initial shape:", df_original.shape)
display(df_original.head())

Initial shape: (442, 11)


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


## 4. Validate and clean the data

We check that `target` exists, convert columns to numeric values, replace infinities with missing values, and remove incomplete rows.

In [5]:
df = df_original.copy()

if "target" not in df.columns:
    raise ValueError("The 'target' column was not found.")

for column in df.columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df = df.replace([np.inf, -np.inf], np.nan)
rows_before = len(df)
df = df.dropna().copy()

print("Rows before cleaning:", rows_before)
print("Rows after cleaning:", len(df))
print("Remaining missing values:", int(df.isna().sum().sum()))

X = df.drop(columns=["target"])
y = df["target"]

print("Number of features:", X.shape[1])
print("Features:", list(X.columns))

Rows before cleaning: 442
Rows after cleaning: 442
Remaining missing values: 0
Number of features: 10
Features: ['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']


## 5. Split the data into training and test sets

We reserve 80% for training and 20% to check how the model predicts on data it did not use during training.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

Training rows: 353
Test rows: 89


## 6. Train Linear Regression and track the experiment

We calculate three metrics:
- **RMSE:** lower is better.
- **MAE:** mean absolute error; lower is better.
- **R²:** indicates how well the predictions fit; it can be negative.

In [7]:
modelo = LinearRegression()

with mlflow.start_run(run_name="LinearRegression") as run:
    modelo.fit(X_train, y_train)
    predicciones = modelo.predict(X_test)

    rmse = float(np.sqrt(mean_squared_error(y_test, predicciones)))
    mae = float(mean_absolute_error(y_test, predicciones))
    r2 = float(r2_score(y_test, predicciones))

    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("random_state", 42)
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("R2", r2)

    # Log the model as an artifact of this run
    mlflow.sklearn.log_model(
        sk_model=modelo,
        artifact_path="model",
        input_example=X_train.head(3)
    )
    run_id = run.info.run_id

print(f"RMSE: {rmse:.2f}")
print(f"MAE:  {mae:.2f}")
print(f"R²:   {r2:.3f}")
print("Run ID:", run_id)

2026/10/10 18:53:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


RMSE: 53.85
MAE:  42.79
R²:   0.453
Run ID: 2f1ab6318a1742aba910ed19be21c4a0


## 7. View the experiment saved in MLflow

This table shows the run we just saved.

In [8]:
runs = mlflow.search_runs(
    experiment_names=["Diabetes-Regression"],
    order_by=["metrics.RMSE ASC"]
)
display(runs[[
    c for c in ["run_id", "metrics.RMSE", "metrics.MAE", "metrics.R2", "params.model_type"]
    if c in runs.columns
]].head())

,run_id,metrics.RMSE,metrics.MAE,metrics.R2,params.model_type
0,2f1ab6318a1742aba910ed19be21c4a0,53.853446,42.794095,0.452603,LinearRegression


## 8. Register the model and assign the `champion` alias

We register the version of the model we just trained. In a real project, the `champion` alias should only be assigned after verifying that the model meets the agreed quality criteria.

In [10]:
from mlflow.tracking import MlflowClient

model_name = "diabetes_model"
model_uri = f"runs:/{run_id}/model"

registered_version = mlflow.register_model(
    model_uri=model_uri,
    name=model_name
)

mlflow_client = MlflowClient()

# Assign the alias to the registered version
mlflow_client.set_registered_model_alias(
    model_name,
    "champion",
    registered_version.version
)

print("Model:", model_name)
print("Version:", registered_version.version)
print("Alias: champion")

Registered model 'diabetes_model' already exists. Creating a new version of this model...
2026/10/10 18:54:06 WARNING mlflow.tracking._model_registry.fluent: Run with id 2f1ab6318a1742aba910ed19be21c4a0 has no artifacts at artifact path 'model', registering model based on models:/m-2253a20cebd445f68560027180d09abf instead


Model: diabetes_model
Version: 2
Alias: champion


Created version '2' of model 'diabetes_model'.


## 9. Check that the `champion` model can be loaded

If this cell runs successfully, MLflow can retrieve the registered model using the alias.

In [11]:
champion_model = mlflow.sklearn.load_model(
    f"models:/{model_name}@champion"
)

example_input = X_test.iloc[[0]]
example_prediction = float(champion_model.predict(example_input)[0])

print("Example prediction:", round(example_prediction, 2))
print("Actual example value:", float(y_test.iloc[0]))

Example prediction: 139.55
Actual example value: 219.0


## 10. Create the API with FastAPI

We create `api.py`. The API provides:
- `GET /health`: checks that the model is loaded.
- `GET /model-info`: returns information about the model.
- `POST /predict`: receives the ten features and returns a prediction.

The API loads the model from MLflow.

In [12]:
%%writefile api.py
import pandas as pd
import mlflow
import mlflow.sklearn

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

mlflow.set_tracking_uri("sqlite:///mlflow.db")

app = FastAPI(title="Diabetes MLOps API")
MODEL_NAME = "diabetes_model"
MODEL_ALIAS = "champion"

try:
    model = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@{MODEL_ALIAS}")
except Exception as error:
    print("Could not load the MLflow model:", error)
    model = None


class PatientData(BaseModel):
    age: float = Field(..., allow_inf_nan=False)
    sex: float = Field(..., allow_inf_nan=False)
    bmi: float = Field(..., allow_inf_nan=False)
    bp: float = Field(..., allow_inf_nan=False)
    s1: float = Field(..., allow_inf_nan=False)
    s2: float = Field(..., allow_inf_nan=False)
    s3: float = Field(..., allow_inf_nan=False)
    s4: float = Field(..., allow_inf_nan=False)
    s5: float = Field(..., allow_inf_nan=False)
    s6: float = Field(..., allow_inf_nan=False)


@app.get("/health")
def health():
    if model is None:
        raise HTTPException(status_code=503, detail="The model is unavailable")
    return {"status": "ok", "model_loaded": True}


@app.get("/model-info")
def model_info():
    if model is None:
        raise HTTPException(status_code=503, detail="The model is unavailable")
    return {
        "model_name": MODEL_NAME,
        "model_alias": MODEL_ALIAS,
        "model_type": "LinearRegression"
    }


@app.post("/predict")
def predict(data: PatientData):
    if model is None:
        raise HTTPException(status_code=503, detail="The model is unavailable")
    try:
        input_data = pd.DataFrame([data.model_dump()])
        prediction = float(model.predict(input_data)[0])
        return {
            "prediction": prediction,
            "model_name": MODEL_NAME,
            "model_alias": MODEL_ALIAS
        }
    except Exception:
        raise HTTPException(status_code=500, detail="Error while making the prediction")

Writing api.py


## 11. Test the endpoints inside Colab

We use `TestClient` to test the API directly without starting an external server.

In [13]:
from fastapi.testclient import TestClient
import importlib
import api

importlib.reload(api)
api_client = TestClient(api.app)

response_health = api_client.get("/health")
print("GET /health:", response_health.status_code, response_health.json())

response_info = api_client.get("/model-info")
print("GET /model-info:", response_info.status_code, response_info.json())

example_input_data = X_test.iloc[0].to_dict()
response_prediction = api_client.post("/predict", json=example_input_data)
print("POST /predict:", response_prediction.status_code, response_prediction.json())

GET /health: 200 {'status': 'ok', 'model_loaded': True}
GET /model-info: 200 {'model_name': 'diabetes_model', 'model_alias': 'champion', 'model_type': 'LinearRegression'}
POST /predict: 200 {'prediction': 139.54755840379607, 'model_name': 'diabetes_model', 'model_alias': 'champion'}


## 12. Create automated tests

The tests use a fake model to test the API quickly and reproducibly. We check a successful request, invalid data, and the case where the model is unavailable.

In [14]:
%%writefile test_api.py
from fastapi.testclient import TestClient
import api

client = TestClient(api.app)

class FakeModel:
    def predict(self, data):
        return [150.0]

def valid_data():
    return {
        "age": 0.04, "sex": 0.05, "bmi": 0.06, "bp": 0.07,
        "s1": 0.08, "s2": 0.09, "s3": 0.10, "s4": 0.11,
        "s5": 0.12, "s6": 0.13
    }

def test_predict_success(monkeypatch):
    monkeypatch.setattr(api, "model", FakeModel())
    response = client.post("/predict", json=valid_data())
    assert response.status_code == 200
    assert response.json()["prediction"] == 150.0

def test_invalid_input():
    response = client.post("/predict", json={"age": "not-a-number"})
    assert response.status_code == 422

def test_model_unavailable(monkeypatch):
    monkeypatch.setattr(api, "model", None)
    response = client.get("/health")
    assert response.status_code == 503

Writing test_api.py


### Run the tests

In [15]:
!pytest -q test_api.py

...                                                                      [100%]
3 passed in 5.13s


## 13. Create `requirements.txt`

In [16]:
%%writefile requirements.txt
mlflow
fastapi
uvicorn
httpx
pytest
scikit-learn
pandas
numpy

Writing requirements.txt


## 14. Set up CI with GitHub Actions

This file makes GitHub install the libraries and run the tests when we push changes or open a pull request. This is continuous integration (CI); it does not automatically deploy the API.

In [17]:
%%writefile .github/workflows/tests.yml
name: Tests MLOps

on:
  push:
  pull_request:

jobs:
  tests:
    runs-on: ubuntu-latest
    steps:
      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt

      - name: Run tests
        run: pytest -v test_api.py

Writing .github/workflows/tests.yml


## 15. Create the project README

In [18]:
%%writefile README.md
# Diabetes dataset with MLOps and Linear Regression

Project Overview

This project implements a simple end-to-end Machine Learning Operations (MLOps) workflow using the Diabetes dataset from scikit-learn.

The goal is to predict diabetes disease progression using patient-related numerical features and a Linear Regression model.

The project demonstrates data preparation, model training, evaluation, experiment tracking, model registration, API serving, automated testing, and continuous integration.

Project Objectives:

Load and validate the dataset.
Clean the data and handle missing or invalid values.
Train a Linear Regression model.
Evaluate model performance using standard regression metrics.
Track experiments using MLflow.
Register the trained model and use the "champion" alias.
Create a REST API using FastAPI.
Test API endpoints and error handling.
Automate tests with GitHub Actions.

Dataset
The project uses the Diabetes dataset provided by scikit-learn.

The dataset contains numerical patient-related features and a target representing a quantitative measure of disease progression.

The model predicts the target value. It does not diagnose diabetes or replace medical advice.

Model and Evaluation
Selected model: Linear Regression.

The model is evaluated using:

Mean Squared Error (MSE): measures the average squared prediction error. Lower values are better.
Root Mean Squared Error (RMSE): measures prediction error in the target's units. Lower values are better.
R² Score: measures how well the model explains variation in the target. Values closer to 1 generally indicate a better fit.
Actual evaluation results are generated when the notebook is executed.

MLOps Workflow

The project follows these steps:

Data extraction and validation.

Data cleaning and preparation.

Train-test split.

Linear Regression model training.

Model evaluation.

Experiment tracking with MLflow.

Model registration and "champion" alias assignment.

Model loading for prediction.

REST API development with FastAPI.

Automated testing with pytest.

Continuous integration with GitHub Actions.

API Endpoints

The API provides the following endpoints:

Method| Endpoint| Description GET| "/health"| Checks service health and model availability. GET| "/model-info"| Returns information about the loaded model. POST| "/predict"| Receives numerical input features and returns a prediction.

The prediction endpoint expects the ten numerical features used by the scikit-learn Diabetes dataset.

Error Handling and Testing

The project includes tests for:

Successful prediction requests.
Invalid input data.
Model or service unavailability.
Tests are implemented using pytest and FastAPI's TestClient.

Technologies Used

Python
pandas
NumPy
scikit-learn
MLflow
FastAPI
Pydantic
pytest
Git and GitHub
GitHub Actions

Repository Structure

diabetes-mlops/ ├── Proyecto_MLOps_Diabetes_Regresion_Lineal.ipynb ├── Diabetes_MLOps_Linear_Regression.ipynb ├── diabetes.csv ├── api.py ├── test_api.py ├── requirements.txt ├── tests.yml └── README.md

The exact filenames and locations should match the files committed to the repository.

Installation and Execution
Install the project dependencies:

pip install -r requirements.txt

Run the notebook in Google Colab, executing the cells in order to prepare the data, train and evaluate the model, and register it with MLflow.

After the model has been registered and the required MLflow tracking environment is available, start the API with:

uvicorn api:app --reload

Open the interactive API documentation at:

"http://127.0.0.1:8000/docs"

Run the automated tests with:

pytest -v

These commands assume the required files, dependencies, and MLflow configuration are available in the current environment.

Continuous Integration

GitHub Actions is used to automate testing when changes are pushed to the repository or a pull request is opened.

The workflow installs the dependencies and runs the automated tests to help identify errors early.

The workflow file must be located inside ".github/workflows/" for GitHub Actions to detect it automatically.

Limitations and Future Improvements

Possible improvements include:

Adding more robust data validation.
Comparing Linear Regression with other regression models.
Improving experiment and model version management.
Persisting MLflow tracking data and model artifacts.
Deploying the API to a hosting platform.
Adding monitoring for prediction latency and model performance.

Conclusion

This project demonstrates the main concepts of an end-to-end MLOps workflow, from data preparation and model training to experiment tracking, API serving, testing, and continuous integration.

It provides an educational foundation for understanding how a machine learning model can be organized and served as part of a software project.

Project type: Educational MLOps project Model: Linear Regression Dataset: scikit-learn Diabetes dataset

Writing README.md


## 16. Download the created files

This cell compresses the code and configuration files so we can download them and upload them to GitHub.

In [19]:
import zipfile

archivos = [
    "api.py",
    "test_api.py",
    "requirements.txt",
    "README.md",
    ".github/workflows/tests.yml"
]

with zipfile.ZipFile("diabetes_mlops_archivos.zip", "w") as zipf:
    for archivo in archivos:
        zipf.write(archivo)

print("Created: diabetes_mlops_archivos.zip")

Created: diabetes_mlops_archivos.zip


## Conclusion

We have built an educational MLOps workflow: data → training → evaluation → MLflow/Champion → API → tests → CI.